<a href="https://colab.research.google.com/github/RajeevKambhatla/longevity-drug-repurposing/blob/main/03_senescence_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gseapy
from google.colab import drive
drive.mount('/content/drive')

import json, pandas as pd, numpy as np
BASE = '/content/drive/MyDrive/longevity_project'
vcap = pd.read_csv(f'{BASE}/data/processed/vcap_sig_info.csv')
genes = pd.read_csv(f'{BASE}/data/raw/GSE92742_Broad_LINCS_gene_info.txt.gz', sep='\t')
print(len(vcap), len(genes))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.7/689.7 kB 8.1 MB/s eta 0:00:00
Mounted at /content/drive
34753 12328


In [2]:
from gseapy import Msigdb
msig = Msigdb()
gmt = msig.get_gmt(category='c2.cgp', dbver='2023.2.Hs')
senmayo = gmt['SAUL_SEN_MAYO']
print(len(senmayo), senmayo[:10])

json.dump(senmayo, open(f'{BASE}/data/processed/senmayo_genes.json', 'w'))

125 ['ACVR1B', 'ANG', 'ANGPT1', 'ANGPTL4', 'AREG', 'AXL', 'BEX3', 'BMP2', 'BMP6', 'C3']


In [3]:
bing = genes[genes['pr_is_bing'] == 1]
id_to_symbol = dict(zip(bing['pr_gene_id'].astype(str), bing['pr_gene_symbol']))
symbols = set(id_to_symbol.values())

in_data = [g for g in senmayo if g in symbols]
print('BING genes:', len(id_to_symbol))
print('SenMayo genes in data:', len(in_data), 'of', len(senmayo))

BING genes: 10174
SenMayo genes in data: 106 of 125


In [4]:
known = ['sirolimus', 'rapamycin', 'everolimus', 'metformin', 'acarbose', 'dasatinib',
         'quercetin', 'fisetin', 'navitoclax', 'resveratrol', 'spermidine', 'nicotinamide']
named = vcap[~vcap['pert_iname'].str.startswith('BRD-')].copy()
named['name'] = named['pert_iname'].str.lower()

known_here = [d for d in known if d in set(named['name'])]
others = named.loc[~named['name'].isin(known_here), 'name'].drop_duplicates()
chosen = known_here + list(others.sample(100 - len(known_here), random_state=42))

pick = named[named['name'].isin(chosen)]
pick = pick.sort_values(['pert_time', 'pert_dose'], ascending=False).groupby('name').head(1)
print('Known drugs included:', known_here)
print('Drugs picked:', len(pick))
pick.to_csv(f'{BASE}/data/processed/senescence_pilot_drugs.csv', index=False)

Known drugs included: ['sirolimus', 'everolimus', 'metformin', 'acarbose', 'dasatinib', 'quercetin', 'navitoclax', 'resveratrol', 'nicotinamide']
Drugs picked: 100


In [5]:
cols = list(id_to_symbol)
sigs = pd.read_parquet(f'{BASE}/data/processed/vcap_signatures.parquet', columns=cols)
sigs = sigs.loc[pick['sig_id']]
sigs.columns = [id_to_symbol[c] for c in sigs.columns]
print(sigs.shape)

(100, 10174)


In [6]:
import gseapy as gp
from tqdm import tqdm

rows = []
for sig_id, row in tqdm(sigs.iterrows(), total=len(sigs)):
    res = gp.prerank(rnk=row.sort_values(ascending=False), gene_sets={'SENMAYO': senmayo},
                     permutation_num=100, min_size=5, max_size=1000,
                     seed=0, outdir=None, verbose=False)
    rows.append({'sig_id': sig_id, 'nes': float(res.res2d['NES'].iloc[0])})

  0%|          | 0/100 [00:00<?, ?it/s]2026-10-03 20:19:13,198 [WARNING] Duplicated values found in preranked stats: 0.09% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  1%|          | 1/100 [00:00<00:15,  6.52it/s]2026-10-03 20:19:13,338 [WARNING] Duplicated values found in preranked stats: 0.06% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  2%|▏         | 2/100 [00:00<00:13,  7.42it/s]2026-10-03 20:19:13,459 [WARNING] Duplicated values found in preranked stats: 0.02% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  3%|▎         | 3/100 [00:00<00:12,  7.62it/s]2026-10-03 20:19:13,588 [WARNING] Duplicated values found in preranked stats: 0.02% of genes
The order of those genes will be arbitrary, which may produce unexpected results.
  4%|▍         | 4/100 [00:00<00:12,  7.50it/s]2026-10-03 20:19:13,724 [WARNING] Duplicated values found in preranked st

In [7]:
scores = pd.DataFrame(rows).merge(pick[['sig_id', 'name', 'pert_dose', 'pert_time']], on='sig_id')
scores['score'] = -scores['nes']
scores = scores.sort_values('score', ascending=False).reset_index(drop=True)
scores['rank'] = scores.index + 1

print(scores.head(10))
print(scores[scores['name'].isin(known_here)])
scores.to_csv(f'{BASE}/results/senescence_baseline_pilot.csv', index=False)

                                         sig_id       nes  \
0     CPC019_VCAP_24H:BRD-A03520402-001-02-1:10 -1.761635   
1     CPC004_VCAP_24H:BRD-A67862938-034-09-9:10 -1.694840   
2     CPC004_VCAP_24H:BRD-A41304429-001-04-9:10 -1.634028   
3  CPC006_VCAP_24H:BRD-K12994359-001-07-7:177.6 -1.587852   
4     CPC004_VCAP_24H:BRD-K47659338-003-01-0:10 -1.573991   
5     CPC014_VCAP_24H:BRD-K18135438-001-08-4:10 -1.466921   
6     CPC002_VCAP_24H:BRD-K23383398-001-01-6:10 -1.456736   
7     CPC004_VCAP_24H:BRD-K51662849-001-03-9:10 -1.451808   
8     CPC010_VCAP_24H:BRD-K13605477-001-03-4:10 -1.438324   
9     CPC004_VCAP_24H:BRD-A75552914-001-09-3:10 -1.383974   

                    name  pert_dose  pert_time     score  rank  
0              st-014366       10.0         24  1.761635     1  
1          naftidrofuryl       10.0         24  1.694840     2  
2              practolol       10.0         24  1.634028     3  
3             valdecoxib      177.6         24  1.587852     4  
4  